# E1 finite native rescoring
Owner: uibcdf/bindingmt#78. Runs native energies only under registered inputs/artifacts and resource limits.
See the dated prescription and separately retained execution evidence.


In [ ]:
import base64
import gzip
import hashlib
import importlib.metadata as metadata
import inspect
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

root = Path(os.environ['BINDINGMT_E1_ROOT'])
evidence = root / 'devguide/validation/data/e1_complete_pose_2026-11-06'
output = Path(os.environ['BINDINGMT_E1_NATIVE_OUTPUT'])
overlay = Path(os.environ['BINDINGMT_E1_VINA_OVERLAY'])
shared = Path('/fast_data/home/liliana/miniconda3/envs/molsyssuite@uibcdf_3.14')
output.mkdir(parents=True, exist_ok=False)
expected_inputs = json.loads(os.environ['BINDINGMT_E1_INPUT_SHA256'])
manifest_bytes = (evidence / 'manifest.json').read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest() == expected_inputs['manifest']
manifest = json.loads(manifest_bytes)
for row in manifest['files']:
    data = (evidence / row['path']).read_bytes()
    assert (
        len(data) == row['bytes'] and hashlib.sha256(data).hexdigest() == row['sha256']
    )
    if row['path'].endswith('.gz'):
        raw = gzip.decompress(data)
        assert (
            len(raw) == row['decoded_bytes']
            and hashlib.sha256(raw).hexdigest() == row['decoded_sha256']
        )
contract_bytes = (evidence / 'prospective_scoring_contract.json').read_bytes()
assert hashlib.sha256(contract_bytes).hexdigest() == expected_inputs['contract']
contract = json.loads(contract_bytes)
assert (
    contract['status'] == 'prospective_not_executed' and contract['version'] == '1.2.7'
)
assert contract['backend_box']['size'] == [24.0, 24.0, 24.0]
staged = output / 'inputs'
staged.mkdir()
receptor = gzip.decompress((evidence / 'provided_receptor.pdbqt.gz').read_bytes())
assert hashlib.sha256(receptor).hexdigest() == contract['receptor_sha256']
receptor_path = staged / 'receptor.pdbqt'
receptor_path.write_bytes(receptor)
ligands = []
for row in contract['submitted_ligands']:
    index = row['original_position']
    data = (evidence / 'normal' / f'pose_{index}' / 'bindingmt.pdbqt').read_bytes()
    assert hashlib.sha256(data).hexdigest() == row['bindingmt_export_sha256']
    path = staged / f'pose_{index}.pdbqt'
    path.write_bytes(data)
    ligands.append(dict(association=row, path=path, content=data))
assert (
    len(ligands) == 5
    and len({row['association']['bindingmt_export_sha256'] for row in ligands}) == 5
)


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.write('\n')


def event(phase, **extra):
    with (output / 'evaluations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    phase=phase, at_utc=datetime.now(timezone.utc).isoformat(), **extra
                ),
                allow_nan=False,
            )
            + '\n'
        )
        stream.flush()


def installed_payload():
    files = {}
    for record_path in sorted((overlay / 'conda-meta').glob('*.json')):
        record = json.loads(record_path.read_text())
        for row in record['paths_data']['paths']:
            path = overlay / row['_path']
            assert path.is_file() and path.resolve().is_relative_to(overlay.resolve())
            digest = sha(path)
            assert digest == row['sha256'], row['_path']
            files[row['_path']] = digest
    return files


payload_before = installed_payload()
shared_meta_before = {p.name: sha(p) for p in (shared / 'conda-meta').glob('*.json')}


sys.path.insert(0, str(overlay / 'lib/python3.14/site-packages'))

In [ ]:
# The loader receives only this artifact overlay; existing runtime stays borrowed.
import molsysmt as msm
import numpy as np
import pyunitwizard as puw
import vina
import vina._vina_wrapper as vina_native
from packaging.version import Version
from smonitor.integrations import CatalogException

import bindingmt as dmt

assert (
    platform.python_version() == '3.14.10'
    and vina.__version__ == metadata.version('vina') == '1.2.7'
)
assert platform.system() == 'Linux' and platform.machine() == 'x86_64'
assert sys.implementation.name == 'cpython' and 't' not in sys.abiflags
libc_name, libc_version = platform.libc_ver()
assert libc_name == 'glibc' and Version('2.17') <= Version(libc_version) < Version('3')
for module in [vina, vina_native]:
    assert Path(module.__file__).resolve().is_relative_to(overlay.resolve())
assert dmt.VinaBackend().is_available
assert inspect.signature(vina.Vina.__init__).parameters['no_refine'].default is False
map_signature = inspect.signature(vina.Vina.compute_vina_maps)
assert map_signature.parameters['spacing'].default == 0.375
assert map_signature.parameters['force_even_voxels'].default is False
versions = {
    json.loads(p.read_text())['name']: json.loads(p.read_text())['version']
    for p in (shared / 'conda-meta').glob('*.json')
}
requirements = {
    'icu': ('78.1', '79'),
    'libgcc': ('14', None),
    'libstdcxx': ('14', None),
    'liblzma': ('5.8.1', '6'),
    'libzlib': ('1.3.1', '2'),
    'bzip2': ('1.0.8', '2'),
    'zstd': ('1.5.7', '1.6'),
}
for name, (lower, upper) in requirements.items():
    assert Version(versions[name]) >= Version(lower)
    if upper is not None:
        assert Version(versions[name]) < Version(upper)
budgets = dict(
    qualification_constructors=2,
    score_attempts=10,
    score_returned=10,
    score_checked=10,
    convert=5,
    get=5,
    audit_pose=10,
    pose_roundtrip=10,
    audit_result=1,
    ranking_refusals=2,
)
counts = dict(
    qualification_constructors=0,
    score_attempts=0,
    score_returned=0,
    score_checked=0,
    convert=0,
    get=0,
    audit_pose=0,
    pose_roundtrip=0,
    audit_result=0,
    ranking_refusals=0,
)
qualification = []
for method in ['vina', 'vinardo']:
    engine = vina.Vina(sf_name=method, cpu=1, seed=7, verbosity=0)
    counts['qualification_constructors'] += 1
    info = engine.info()
    assert list(info['weights']) == contract['expected_factory_weights'][method]
    qualification.append(
        dict(method=method, constructor=dict(cpu=1, seed=7), info=info)
    )
    del engine


def native_inventory():
    paths = set()
    for line in Path('/proc/self/maps').read_text().splitlines():
        fields = line.split(None, 5)
        if len(fields) == 6 and fields[5].startswith('/') and '.so' in fields[5]:
            path = Path(fields[5])
            if path.is_file():
                paths.add(path.resolve())
    rows = []
    for path in sorted(paths):
        if path.is_relative_to(overlay.resolve()):
            location = 'engine_overlay'
            label = str(path.relative_to(overlay.resolve()))
        elif path.is_relative_to(shared):
            location = 'borrowed_runtime'
            label = str(path.relative_to(shared))
        elif path.is_relative_to(root.parent):
            location = 'editable_provider'
            label = str(path.relative_to(root.parent))
        else:
            location = 'system'
            label = str(path)
        rows.append(dict(location=location, path=label, sha256=sha(path)))
    return rows


native_before = native_inventory()
boost_loaded = [row for row in native_before if 'libboost_' in row['path']]
assert boost_loaded and all(
    row['location'] == 'engine_overlay' and '1.86.0' in row['path']
    for row in boost_loaded
)
save(
    output / 'backend_qualification.json',
    dict(
        scope='Borrowed source runtime, selected native artifact and actual loaded libraries; no independent energy accuracy or public closure claim',
        python=platform.python_version(),
        versions={
            name: metadata.version(name)
            for name in [
                'vina',
                'bindingmt',
                'molsysmt',
                'argdigest',
                'pyunitwizard',
                'numpy',
                'depdigest',
                'smonitor',
            ]
        },
        installed_payload_sha256=payload_before,
        native_libraries=native_before,
        probes=qualification,
        installed_defaults=dict(
            no_refine=False, spacing_angstrom=0.375, force_even_voxels=False
        ),
        native_source_build_correspondence='unassessed',
        borrowed_dependency_versions={name: versions[name] for name in requirements},
    ),
)

In [ ]:
components = [
    'total',
    'lig_inter',
    'flex_inter',
    'other_inter',
    'flex_intra',
    'lig_intra',
    'torsions',
    'lig_intra_best_pose',
]
caller_policy = puw.configure.report()
final_poses = []
observations = []
with puw.context(
    standard_units=[
        'pm',
        'fs',
        'kelvin',
        'mole',
        'dalton',
        'coulomb',
        'kJ/mol',
        'kJ/(mol*pm)',
        'kJ/(mol*pm**2)',
        'radians',
    ]
):
    active_policy = puw.configure.report()
    box = dmt.BoxRegion(
        puw.quantity(contract['backend_box']['center'], 'angstrom'),
        puw.quantity(contract['backend_box']['size'], 'angstrom'),
    )
    for item in ligands:
        association = item['association']
        index = association['original_position']
        directory = output / f'pose_{index}'
        directory.mkdir()
        native = msm.convert(
            item['path'], to_form='molsysmt.MolSys', discard_torsion_tree=True
        )
        counts['convert'] += 1
        xyz = msm.get(native, element='atom', coordinates=True)
        counts['get'] += 1
        assert xyz.shape == (1, 13, 3)
        receptor_state = 'e1-captured-3PTB-receptor-20261020'
        partner_state = association['new_partner_state_id']
        problem = dmt.DockingProblem(
            receptor=receptor_path,
            partner=item['path'],
            search_domain=box,
            metadata=dict(
                receptor_state_id=receptor_state, partner_state_id=partner_state
            ),
        )
        pose = dmt.DockingPose(
            xyz[0],
            pose_id=association['new_pose_id'],
            receptor_state_id=receptor_state,
            partner_state_id=partner_state,
            metadata=dict(
                pose_atom_order='pdbqt_record_order',
                historical_association=association,
                construction_manifest_sha256=expected_inputs['manifest'],
            ),
        )
        assert pose.rank is None and not pose.scores
        for method in ['vina', 'vinardo']:
            before = pose.to_dict()
            protocol = dmt.VinaProtocol(
                scoring=method,
                cpu=1,
                seed=7,
                capture_backend_inputs=True,
                collect_timings=True,
                allow_provisional_preparation=False,
            )
            event(
                'started',
                position=index,
                method=method,
                attempt=counts['score_attempts'] + 1,
            )
            assert counts['score_attempts'] < 10
            counts['score_attempts'] += 1
            scored = dmt.score(
                problem, protocol, backend='vina', pose=pose, score_name=method
            )
            counts['score_returned'] += 1
            record = scored.to_dict()
            save(directory / (method + '_scored_pose.json'), record)
            event(
                'returned',
                position=index,
                method=method,
                record=method + '_scored_pose.json',
            )
            assert pose.to_dict() == before
            assert (
                record['pose_id'] == before['pose_id']
                and record['rank'] == before['rank']
            )
            assert record['coordinates'] == before['coordinates']
            assert (
                record['partner_state_id'] == partner_state
                and record['receptor_state_id'] == receptor_state
            )
            for name, value in before['scores'].items():
                assert record['scores'][name] == value
            names = [method] + [method + '.' + c for c in components[1:]]
            assert len(set(record['scores']) - set(before['scores'])) == 8
            assert all(
                type(record['scores'][name]) is float
                and np.isfinite(record['scores'][name])
                for name in names
            )
            history = record['metadata']['scoring_history']
            evaluation = history[-1]
            assert len(history) == (1 if method == 'vina' else 2)
            if method == 'vinardo':
                assert history[:-1] == before['metadata']['scoring_history']
            assert (
                evaluation['backend_version'] == '1.2.7'
                and evaluation['operation'] == 'score'
            )
            assert evaluation['unused_search_parameters'] == [
                'exhaustiveness',
                'n_poses',
                'energy_range',
            ]
            assert (
                evaluation['applied_parameters']['cpu'] == 1
                and evaluation['applied_parameters']['seed'] == 7
            )
            assert evaluation['applied_parameters']['scoring'] == method
            assert evaluation['protocol']['parameters']['active_torsion_bonds'] == []
            artifacts = evaluation['backend_artifacts']
            dmt.verify_captured_inputs(artifacts)
            for role, expected in [
                ('receptor', receptor),
                ('partner', item['content']),
            ]:
                assert (
                    base64.b64decode(artifacts[role]['content_base64'], validate=True)
                    == expected
                )
                assert artifacts[role]['sha256'] == hashlib.sha256(expected).hexdigest()
            for name, component in zip(names, components):
                definition = record['metadata']['score_definitions'][name]
                context = definition['context']
                assert (
                    definition['method'] == f'AutoDock Vina/{method}'
                    and definition['method_version'] == '1.2.7'
                    and definition['component'] == component
                )
                assert puw.are_compatible(
                    puw.quantity(1, definition['unit']), 'kcal/mol'
                )
                assert (
                    context['partner_sha256'] == association['bindingmt_export_sha256']
                    and context['receptor_sha256'] == contract['receptor_sha256']
                )
                assert (
                    context['backend_box'] == contract['backend_box']
                    and context['grid_spacing'] == contract['grid_spacing']
                )
                assert (
                    context['weights'] == contract['expected_factory_weights'][method]
                )
                assert context['preparation_assessment'] == dict(
                    receptor='unassessed', partner='unassessed'
                )
                assert (
                    context['reference_energy_policy'] == 'native_current_conformation'
                )
            reference = evaluation['reference_energy']
            assert reference == dict(
                schema_version='1.0',
                policy='native_current_conformation',
                score_name=method + '.lig_intra_best_pose',
                value=record['scores'][method + '.lig_intra_best_pose'],
                unit='kcal/mol',
            )
            audit = dmt.audit_pose(record)
            counts['audit_pose'] += 1
            save(directory / (method + '_audit.json'), audit)
            assert audit['status'] == 'consistent'
            roundtrip = dmt.DockingPose.from_dict(
                json.loads(json.dumps(record, allow_nan=False))
            ).to_dict()
            counts['pose_roundtrip'] += 1
            assert roundtrip == record
            assert puw.configure.report() == active_policy
            counts['score_checked'] += 1
            event('checked', position=index, method=method)
            observations.append(
                dict(
                    position=index,
                    pose_id=record['pose_id'],
                    method=method,
                    unit='kcal/mol',
                    scores={name: record['scores'][name] for name in names},
                    reference=reference,
                    partner_sha256=association['bindingmt_export_sha256'],
                    receptor_sha256=contract['receptor_sha256'],
                )
            )
            pose = scored
        assert len(pose.scores) == 16
        final_poses.append(pose)
assert puw.configure.report() == caller_policy

In [ ]:
assert (
    counts['score_attempts']
    == counts['score_returned']
    == counts['score_checked']
    == 10
)
assert len(final_poses) == 5 and len(observations) == 10
collection = dmt.DockingResult(poses=final_poses)
collection_record = collection.to_dict()
save(output / 'scored_collection.json', collection_record)
collection_audit = dmt.audit_result(collection_record)
counts['audit_result'] += 1
save(output / 'collection_audit.json', collection_audit)
assert collection_audit['status'] == 'incomplete'
assert not any(row['status'] == 'inconsistent' for row in collection_audit['checks'])
refusals = []
for method in ['vina', 'vinardo']:
    try:
        collection.rank_by(method)
    except CatalogException as error:
        assert 'identical declared definitions and contexts' in str(error)
        counts['ranking_refusals'] += 1
        refusals.append(
            dict(
                method=method,
                code=getattr(error, 'code', None),
                exception_type=type(error).__name__,
                message=str(error),
            )
        )
    else:
        raise AssertionError('Expected strict ranking refusal')
assert collection.to_dict() == collection_record
save(output / 'ranking_refusals.json', refusals)
assert installed_payload() == payload_before
assert {
    p.name: sha(p) for p in (shared / 'conda-meta').glob('*.json')
} == shared_meta_before
assert receptor_path.read_bytes() == receptor
for row in ligands:
    assert row['path'].read_bytes() == row['content']
assert sha(evidence / 'manifest.json') == expected_inputs['manifest']
assert (
    sha(evidence / 'prospective_scoring_contract.json') == expected_inputs['contract']
)
assert counts == dict(
    qualification_constructors=2,
    score_attempts=10,
    score_returned=10,
    score_checked=10,
    convert=5,
    get=5,
    audit_pose=10,
    pose_roundtrip=10,
    audit_result=1,
    ranking_refusals=2,
)
save(
    output / 'observations.json',
    dict(
        scope='New empirical fixed-score observations, in original pose/method loop order; no numerical ranking or historical-energy equivalence.',
        values=observations,
    ),
)
save(
    output / 'receipt.json',
    dict(
        schema='bindingmt.e1_native_rescoring_receiving@1',
        owner='uibcdf/bindingmt#78',
        counts=counts,
        completed_poses=5,
        new_empirical_score_values=80,
        unit_policy='One nondefault pm/fs/coulomb/kJ-per-mol native population; backend fields retain explicit angstrom/kcal-per-mol.',
        input_sha256=expected_inputs,
        standalone_audits='consistent',
        collection_audit='incomplete_without_collection_stage_provenance',
        ranking='strictly refused, no ordering performed',
        native_libraries_after=native_inventory(),
        engine_payload_unchanged=True,
        shared_conda_metadata_unchanged=True,
        native_source_build_correspondence='unassessed',
        observations_file='observations.json',
        scope='Borrowed development source/native integration, not ordinary-installed/public closure or independent energy accuracy.',
        full_e1_accepted=False,
        historical_prepared_H_identity_admitted=False,
        shared_reference_comparison_admitted=False,
    ),
)
print(
    json.dumps(dict(score_attempts=10, checked=10, new_values=80, ranking_refusals=2))
)